# 02 — Supervisado: qué representación, qué reducción y con qué umbral

Decide tres cosas, en ese orden. No entrena el modelo de producción.

**Regla de oro:** el reductor se ajusta **solo con el train de cada fold**. No hay imputador: el Random Forest de `sklearn` maneja los NaN de forma nativa.


## 0. Entorno

Una sola variable de ruta. Los escenarios se cargan por separado y nunca se mezclan.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src2.data import eliminar_columnas
from src2.features import rango_escenarios, rango_movil
from Colab.papelera.reduce import REDUCTORES
from src2.split import separar_variables, aplanar_escenarios, aplanar_ventanas
from src2.train import (especificidad_s0, resumen_configuraciones, rf_CV,
                        rf_LOSO)

pd.set_option("display.width", 180)

BASE = Path("/media/windows/Users/pablo/Documents/UNSA/4to/MetodosCuantitativos/TPFinal/tpfinal-colab/datasets/")

ARCHIVOS = {
    "S1": "datasetS1.csv", "S1alt": "datasetS1alt.csv", "S2": "datasetS2.csv",
    "S3": "datasetS3.csv", "S4": "datasetS4.csv", "S5": "datasetS5.csv",
    "S6": "datasetS6.csv",
}


In [ ]:
brutos = {k: pd.read_csv(BASE / f) for k, f in ARCHIVOS.items()}
s0_bruto = pd.read_csv(BASE / "datasetS0.csv")

dfs = {k: eliminar_columnas(df) for k, df in brutos.items()}
s0 = eliminar_columnas(s0_bruto)

X, Y = {}, {}
for k, df in dfs.items():
    X[k], Y[k] = separar_variables(df)

X_s0, _ = separar_variables(s0)
print("escenarios:", list(dfs), "| features:", X["S1"].shape[1])


## 1. La fuga de `Profile.*`

`Profile.*` no es una medición: es el guion del escenario. Si S0 se grabó con el perfil congelado y
los escenarios de ataque con el perfil variable, la columna identifica el escenario. Por eso
`eliminar_columnas` la descarta.


In [ ]:
perfil = {}
for nombre, df in {**brutos, "S0": s0_bruto}.items():
    perfil[nombre] = {
        "desvio": df["Profile.pv_active_kw"].std(),
        "minimo": df["Profile.pv_active_kw"].min(),
        "maximo": df["Profile.pv_active_kw"].max(),
    }
display(pd.DataFrame(perfil).T.round(4))


## 2. Protocolo

**Por qué LOSO.** Un split aleatorio en datos de 1 Hz es una trampa: dos filas consecutivas son casi
la misma fila, así que el test queda lleno de copias del entrenamiento. La comparación de abajo lo
muestra con el mismo modelo y las mismas filas.

**Con qué se mide.** AUC por fold y `recall@5` (fracción de ataques en el 5 % de mayor score, la
convención de `src/train.py`). La especificidad se informa a TPR fijo, no a 0.5: un modelo puede
ordenar bien y fallar el corte.


In [ ]:
tabla_cv = rf_CV(X, Y, 8)
tabla_loso = rf_LOSO(X, Y)

print("AUC medio, CV aleatoria:", round(tabla_cv.loc["mean", "auc"], 3))
print("AUC medio, LOSO:", round(tabla_loso.auc.mean(), 3))
display(tabla_loso)


## 3. Las tres representaciones

- **crudo**: una fila es un segundo, 132 features.
- **rango**: una fila es un segundo, 132 features de amplitud.
- **ventana**: **una muestra son 20 segundos**, 132 × 20 = 2640 features aplanadas. Pierde las
  primeras 19 filas de cada escenario y hereda la etiqueta de su fila de cierre.


In [ ]:
X_rango = rango_escenarios(X)
X_ventana, Y_ventana = aplanar_escenarios(X, Y, 20)

X_s0_rango = rango_movil(X_s0)
y_s0 = pd.Series(0, index=X_s0.index)
X_s0_ventana, _ = aplanar_ventanas(X_s0, y_s0, 20)


In [ ]:
tabla_crudo = rf_LOSO(X, Y)
tabla_rango = rf_LOSO(X_rango, Y)
tabla_ventana = rf_LOSO(X_ventana, Y_ventana)

espec_crudo = especificidad_s0(X, Y, X_s0)
espec_rango = especificidad_s0(X_rango, Y, X_s0_rango)
espec_ventana = especificidad_s0(X_ventana, Y_ventana, X_s0_ventana)

resumen = resumen_configuraciones(
    {"crudo": tabla_crudo, "rango": tabla_rango, "ventana": tabla_ventana},
    {"crudo": espec_crudo, "rango": espec_rango, "ventana": espec_ventana},
)
display(resumen)


### 3.1 El reloj del dispositivo

`Substation-AC-Meter.realtime` es el reloj interno del equipo. Aplanado en 20 pasos entrega 20 valores
que codifican la posición de la muestra en la grabación. La prueba es una sola columna: la misma
ventana con el contador y sin él.


In [ ]:
X_reloj = {k: X[k].assign(realtime=brutos[k]["Substation-AC-Meter.realtime"]) for k in X}
s0_reloj = X_s0.assign(realtime=s0_bruto["Substation-AC-Meter.realtime"])

X_ventana_reloj, Y_ventana_reloj = aplanar_escenarios(X_reloj, Y, 20)
X_s0_ventana_reloj, _ = aplanar_ventanas(s0_reloj, y_s0, 20)

tabla_reloj = rf_LOSO(X_ventana_reloj, Y_ventana_reloj)
espec_reloj = especificidad_s0(X_ventana_reloj, Y_ventana_reloj, X_s0_ventana_reloj)

print("ventana sin reloj:", round(tabla_ventana.auc.mean(), 3), "| espec S0", round(espec_ventana, 3))
print("ventana con reloj:", round(tabla_reloj.auc.mean(), 3), "| espec S0", round(espec_reloj, 3))


## 4. Reducción

Cuatro opciones, todas ajustadas **con el train de cada fold**:

- `ninguna`: las 132 columnas.
- `corr`: descarta una de cada par con |r| > 0.95.
- `pca`: imputa, escala y proyecta conservando el 95 % de la varianza.
- `block_pca`: agrupa por correlación y aplica un PCA por bloque.

Se corre sobre el rango porque es la representación que ganó la sección 3: mejor AUC, mejor
recall@5 y el desvío más chico entre escenarios.

Esta tabla **no se puede trasladar a otra representación**: el reductor no es neutral. Un `block_pca`
ajustado sobre amplitudes no significa lo mismo sobre niveles crudos, y el ranking cambia.


In [ ]:
# --- cuatro reducciones sobre el rango, mismo modelo y mismos folds ---
tablas, especificidades = {}, {}
for nombre, reductor in REDUCTORES.items():
    tablas[nombre] = rf_LOSO(X_rango, Y, reductor)
    especificidades[nombre] = especificidad_s0(X_rango, Y, X_s0_rango, reductor=reductor)

display(resumen_configuraciones(tablas, especificidades))


## Lo que falta

- **4. Reducción** sobre la representación elegida: `ninguna`, `corr`, `pca`, `block_pca`, ajustados
  por fold, con la tabla de interacción sobre las otras dos representaciones.
- **5. Modelo**: Random Forest contra XGBoost sobre la configuración elegida.
- **6. Punto de operación**: umbral por fold a TPR objetivo, especificidad S0 a 90/95/99 %.
- **7. Auditoría**: validación anidada, porque elegir el máximo entre varias configuraciones sobre
  7 folds es optimista.
- **8. Producción**: reentrenar con todo, guardar y predecir un escenario nuevo.
